<a href="https://colab.research.google.com/github/lawrence-kagugo/kenya-telecom-churn-analysis/blob/main/notebooks/05_statistical_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kenya Telecom Customer Churn - Statistical Analysis
**Step:** 9 - Statistical Analysis

In [1]:
import pandas as pd
from scipy import stats

url = "https://raw.githubusercontent.com/lawrence-kagugo/kenya-telecom-churn-analysis/refs/heads/main/data/processed/kenya_telecom_churn_cleaned.csv"
df = pd.read_csv(url)
print("Loaded:", df.shape)

Loaded: (5000, 70)


In [2]:
# Independent t-test: compares the means of SupportTickets between two
# independent groups (churned vs retained). Returns a t-statistic (size/
# direction of difference relative to variability) and a p-value (how likely
# this result is under "no real difference").
retained = df[df['Churn'] == 'No']['SupportTickets']
churned = df[df['Churn'] == 'Yes']['SupportTickets']

t_stat, p_value = stats.ttest_ind(retained, churned)
print(f"T-statistic: {t_stat:.3f}")
print(f"P-value: {p_value:.10f}")

T-statistic: -20.089
P-value: 0.0000000000


## Test 1: Support Tickets (t-test)

**Result:** t = -20.089, p < 0.0001 (effectively 0)

**Conclusion:** Statistically significant. The difference in SupportTickets
between churned and retained customers is real, not due to chance.

In [3]:
retained_esc = df[df['Churn'] == 'No']['Escalations']
churned_esc = df[df['Churn'] == 'Yes']['Escalations']

t_stat, p_value = stats.ttest_ind(retained_esc, churned_esc)
print(f"T-statistic: {t_stat:.3f}")
print(f"P-value: {p_value:.10f}")

T-statistic: -9.933
P-value: 0.0000000000


## Test 2: Escalations (t-test)

**Result:** t = -9.933, p < 0.0001

**Conclusion:** Statistically significant - the difference is real, not random
noise. However, the absolute effect is small (0.017 vs 0.074 escalations per
customer) - this is a case where statistical significance does NOT imply
strong practical importance. With 5,000 rows, even small real differences
reach significance. Escalations should be treated as a weak-to-moderate
driver, not a primary one, despite the significant p-value.

In [4]:
# Chi-square test of independence: checks whether ContractType and Churn
# are related, or whether churn is evenly distributed across contract types
# (independence). We first build a "contingency table" - counts of each
# ContractType x Churn combination.
contingency = pd.crosstab(df['ContractType'], df['Churn'])
print(contingency)
print()

chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
print(f"Chi-square statistic: {chi2:.3f}")
print(f"P-value: {p_value:.10f}")
print(f"Degrees of freedom: {dof}")

Churn             No  Yes
ContractType             
12 Months       1235  228
24 Months        690  114
Month-to-Month  2003  730

Chi-square statistic: 100.022
P-value: 0.0000000000
Degrees of freedom: 2


## Test 3: Contract Type vs Churn (Chi-square)

**Result:** χ² = 100.022, p < 0.0001, df = 2

**Conclusion:** Statistically significant association between ContractType
and Churn. This confirms Q1's finding is not due to chance - contract
structure is a genuine, defensible churn driver.

In [5]:
# Run t-tests on all remaining numeric drivers from Q5 (engagement) and
# Q6 (satisfaction) in one pass - same test, applied systematically.
metrics_to_test = ['LoginsPerMonth', 'SessionsPerWeek', 'FeatureAdoptionRate',
                    'NetPromoterScore', 'CustomerSatisfactionScore',
                    'SurveyScore', 'ProductRating', 'LatePayments', 'OutstandingBalance']

results = []
for col in metrics_to_test:
    retained_vals = df[df['Churn'] == 'No'][col]
    churned_vals = df[df['Churn'] == 'Yes'][col]
    t_stat, p_val = stats.ttest_ind(retained_vals, churned_vals)
    results.append({'Metric': col, 'T-statistic': round(t_stat, 3), 'P-value': p_val, 'Significant (p<0.05)': p_val < 0.05})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

                   Metric  T-statistic      P-value  Significant (p<0.05)
           LoginsPerMonth        8.894 8.089781e-19                  True
          SessionsPerWeek        7.288 3.625385e-13                  True
      FeatureAdoptionRate        7.981 1.785858e-15                  True
         NetPromoterScore        7.503 7.322337e-14                  True
CustomerSatisfactionScore       14.618 2.024178e-47                  True
              SurveyScore       12.914 1.481500e-37                  True
            ProductRating       10.758 1.063335e-26                  True
             LatePayments        1.566 1.175145e-01                 False
       OutstandingBalance       -1.879 6.034713e-02                 False


## Tests 4-10: Engagement, Satisfaction, and Billing metrics (t-tests)

**Results:**
- LoginsPerMonth, SessionsPerWeek, FeatureAdoptionRate (Q5 engagement): all
  p < 0.0001 - statistically significant.
- NetPromoterScore, CustomerSatisfactionScore, SurveyScore, ProductRating
  (Q6 satisfaction): all p < 0.0001 - statistically significant.
- LatePayments (p = 0.118) and OutstandingBalance (p = 0.060): NOT
  significant at the 0.05 threshold.

**Conclusion:** Statistical testing confirms the EDA pattern exactly -
engagement and satisfaction are genuine, well-evidenced churn drivers.
Billing behavior is confirmed as NOT a statistically reliable driver,
giving formal backing to the Q4 finding rather than relying on eyeballed
averages alone.

In [6]:
retained_tenure = df[df['Churn'] == 'No']['TenureMonths']
churned_tenure = df[df['Churn'] == 'Yes']['TenureMonths']

t_stat, p_value = stats.ttest_ind(retained_tenure, churned_tenure)
print(f"T-statistic: {t_stat:.3f}")
print(f"P-value: {p_value:.10f}")

T-statistic: 23.503
P-value: 0.0000000000


In [7]:
contingency_seg = pd.crosstab(df['CustomerSegment'], df['Churn'])
print(contingency_seg)
print()

chi2, p_value, dof, expected = stats.chi2_contingency(contingency_seg)
print(f"Chi-square statistic: {chi2:.3f}")
print(f"P-value: {p_value:.10f}")
print(f"Degrees of freedom: {dof}")

Churn              No  Yes
CustomerSegment           
Corporate         354  193
Government        158   94
Residential      2265  459
SME               737  244
Student           414   82

Chi-square statistic: 147.903
P-value: 0.0000000000
Degrees of freedom: 4


## Test 11: Tenure (t-test)

**Result:** t = 23.503, p < 0.0001

**Conclusion:** Statistically significant - the strongest t-statistic of any
metric tested, confirming Tenure as the most robust churn driver identified
in this analysis.

## Test 12: Customer Segment vs Churn (Chi-square)

**Result:** χ² = 147.903, p < 0.0001, df = 4

**Conclusion:** Statistically significant association between CustomerSegment
and Churn - stronger than ContractType's association (χ²=100.0). Confirms
Q8's segment-level churn rate differences are real, not sampling noise.

## Statistical Analysis Summary

| Driver | Test | Significant? | Strength |
|---|---|---|---|
| Contract Type | Chi-square | Yes (p<0.0001) | Strong |
| Tenure | T-test | Yes (p<0.0001) | Strong |
| Support Tickets | T-test | Yes (p<0.0001) | Moderate |
| Escalations | T-test | Yes (p<0.0001) | Weak (significant but tiny absolute effect) |
| Engagement (3 metrics) | T-test | Yes, all 3 | Strong |
| Satisfaction (4 metrics) | T-test | Yes, all 4 | Strong |
| Billing (2 metrics) | T-test | No, both | Not a driver |
| Customer Segment | Chi-square | Yes (p<0.0001) | Strong |

**Key conclusion:** Tenure, Contract Type, Customer Segment, Engagement, and
Satisfaction are all statistically validated, strong churn drivers. Support
metrics are moderate-to-weak (statistically real but smaller practical effect).
Billing behavior is confirmed NOT to be a reliable churn driver in this
dataset - this was hypothesized as weak in Q4 and is now formally confirmed
rather than assumed.